# NB 2.2 &mdash; Regressió lineal múltiple: afegir variables d'una en una

**MP 5134** &mdash; UT2 &mdash; *Versió California Housing*

*Dades: California Housing, districtes censals de Califòrnia (cens de 1990).*

---
### Què farem avui

Al NB 2.1 vam predir la massa d'un pingüí amb una sola mesura. Avui canviem de
dades i de problema: deixem l'Antàrtida i passem al **preu dels habitatges de
Califòrnia**. Farem servir vint mil zones del cens dels Estats Units per predir
quant valen les cases de cada zona, i ens farem una pregunta molt pràctica: **si
li donem més variables al model, prediu millor el preu?** I si és així, quant
millor?

Aquestes dades ens acompanyaran bona part del curs, així que avui també les
repartirem en trossos, cadascun amb una feina diferent. Un d'aquests trossos
quedarà guardat sota clau fins a final de curs.

Quan acabis hauries de saber respondre aquestes preguntes:

- Què conté el fitxer de California Housing i quines sorpreses amaga?
- Per què apartem un tros de les dades i no el mirem fins a final de curs?
- Per què necessitem un conjunt de *validació* a més del d'entrenament?
- Afegir variables sempre ajuda? Quant? Importa l'ordre?
- Com es llegeixen els coeficients d'un model amb moltes variables, i quins
  paranys tenen?

Pel camí trobaràs tres pauses **Ara tu**, com al NB 2.1. Són exercicis curts,
lligats a la secció que acabes de veure, perquè provis amb les teves mans el que
s'acaba d'explicar abans de continuar.

## 1. Les dades: vint mil districtes de Califòrnia

**California Housing** és un conjunt de dades construït a partir del **cens dels
Estats Units de 1990**. És el que fa servir Aurélien Géron per obrir el seu llibre
d'aprenentatge automàtic, i s'ha convertit en un clàssic per una raó: és prou
senzill per entendre'l d'una ullada i prou brut per ensenyar els problemes de les
dades reals.

Abans de res, una cosa que convé tenir clara: **cada fila no és un habitatge**,
sinó un *districte censal*, una zona d'entre 600 i 3.000 habitants. Tots els valors
són del districte sencer: el total d'habitacions de totes les cases, els ingressos
mitjans de totes les llars, el preu mitjà de tots els habitatges.

El fitxer és públic, al repositori del llibre, i el carreguem directament per URL.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# California Housing: districtes censals de Califòrnia (cens de 1990)
URL_DADES = ("https://raw.githubusercontent.com/ageron/handson-ml2/"
             "master/datasets/housing/housing.csv")

df = pd.read_csv(URL_DADES)

print(f"Districtes al fitxer: {len(df)}")
df.head()

Més de vint mil districtes, i cada fila té deu columnes:

| Columnes | Què són |
|---|---|
| `longitude`, `latitude` | on és el districte |
| `housing_median_age` | antiguitat mediana dels habitatges, en anys |
| `total_rooms`, `total_bedrooms` | habitacions i dormitoris **de tot el districte** |
| `population`, `households` | habitants i llars del districte |
| `median_income` | ingressos mitjans de les llars, en **desenes de milers** de dòlars (un 3,5 vol dir 35.000 \$ l'any) |
| `ocean_proximity` | proximitat al mar, en text |
| `median_house_value` | **preu mitjà dels habitatges, en dòlars**: el que volem predir |

Que siguin totals de districte té conseqüències: un districte gran tindrà moltes
habitacions simplement perquè hi viu més gent, no perquè les cases siguin més
grans. Ho tornaràs a trobar a la secció 8.

Ara mirem com és el fitxer per dins: de quin tipus és cada columna i si hi falta
alguna dada.

In [ ]:
df.info()

`info()` dóna el tipus de cada columna i quants valors no buits té. Hi ha dues coses
per fixar-s'hi:

- **`ocean_proximity` és text** (`object`). Els models de scikit-learn només
  accepten números, així que avui la deixarem fora de les entrades. Convertir-la en
  números és feina de la **UT3**.
- **`total_bedrooms` té menys valors que la resta**: 20.433 de 20.640. Hi ha 207
  districtes on no sabem quants dormitoris hi ha. Els tornarem a trobar de seguida.

Fins aquí hem mirat *com és* el fitxer. Per mirar *què hi ha a dins* (com es
reparteixen els preus, on són els districtes cars) esperarem a la secció 3, després
d'apartar un tros de les dades que no hem de veure. De seguida entendràs per què.

## 2. Una part de les dades, sota clau

Abans d'entrenar res farem una cosa que només es fa una vegada en tot el curs:
apartar un tros de les dades i no mirar-lo fins al final. És el que anomenarem el
**test segellat**.

Durant les pròximes unitats entrenarem desenes de models sobre aquestes dades, i el
test segellat es quedarà tancat tot aquest temps. L'obrirem tots junts a classe a
la **UT10**, amb el projecte, per comprovar si els models que haurem construït
funcionen de veritat amb districtes que no han vist mai.

Per què tant de misteri, si a la UT1 vam fer una partició d'entrenament i prova i
prou? Ho entendràs al final del NB 2.3, quan hàgim triat un model mirant-ne els
resultats. De moment, queda't amb la idea: **és l'examen final, i ningú no el pot
veure abans d'hora.**

In [ ]:
from sklearn.model_selection import train_test_split

dades, test_segellat = train_test_split(df, test_size=0.2, random_state=42)

print(f"Districtes que ens quedem per treballar: {len(dades)}")
print(f"Districtes del test segellat:            {len(test_segellat)}  ({100 * len(test_segellat) / len(df):.0f} % del total)")

del test_segellat   # des d'aquí, en aquest notebook ja no existeix

Un 20% de les dades, dins del marge habitual d'entre el 10% i el 30%. La línia
`del test_segellat` esborra la variable: és un gest simbòlic, però fa que sigui
impossible fer-la servir per descuit més avall.

Aquí la peça clau és el **`random_state=42`**. `train_test_split` tria els
districtes a l'atzar, i el 42 fa que cada vegada triï exactament els mateixos. Si
algú el canviés, el test serien uns altres districtes, i alguns d'ells ja els
hauríem fet servir per entrenar. **Aquest 42 és el pany del segell, i no es toca
mai.** A partir d'ara, tots els notebooks del curs que facin servir aquestes dades
començaran amb aquestes mateixes línies.

Potser et preguntes per què aquí triem a l'atzar, si en un problema de predir el
futur caldria separar per dates. És que aquí **no hi ha dates**: tot el fitxer és
una fotografia del cens de 1990. Cada districte és un cas diferent, i triar-los a
l'atzar imita la situació real, que és fer servir el model amb districtes que no
ha vist.

Això sí, amb un matís que convé no oblidar. Ho veuràs al mapa de la secció 3:
**els districtes veïns s'assemblen molt**. Tallant a l'atzar, un districte de la prova
sovint té el seu veí de carrer a l'entrenament, i això ajuda una mica el model.
Els números que obtindrem seran lleugerament optimistes. A la **UT7** veurem
maneres de partir les dades que ho tenen en compte.

## 3. Ara sí, mirem les dades

Amb el test segellat apartat, ja podem explorar sense por. **Tot el que mirem a
partir d'ara és sobre `dades`**, el 80% que ens queda. Abans de fer cap model, mirem
les dades, i comencem per la que volem predir: el preu.

In [ ]:
plt.figure(figsize=(9, 4))
plt.hist(dades["median_house_value"], bins=60, edgecolor="white")
plt.xlabel("Preu mitjà dels habitatges del districte ($)")
plt.ylabel("Nombre de districtes")
plt.title("Com es reparteixen els preus?")
plt.show()

topall = dades["median_house_value"].max()
al_topall = (dades["median_house_value"] == topall).sum()
print(f"Preu màxim: {topall:,.0f} $")
print(f"Districtes amb aquest preu exacte: {al_topall} ({100 * al_topall / len(dades):.1f} %)")

Un **histograma** reparteix els preus en intervals i compta quants districtes cauen
a cadascun. La meitat dels districtes valen entre 120.000 i 265.000 \$, i hi ha
una cua llarga cap als preus alts.

I a la dreta de tot hi ha la **primera sorpresa**: una barra aïllada, molt més alta
del que li tocaria. Gairebé un 5% dels districtes valen *exactament* 500.001 \$, que
és el màxim. No és casualitat: quan es van recollir les dades, **es va decidir
tallar els preus** per sobre de mig milió. Un districte que valia 800.000 \$ apareix
al fitxer com a 500.001 \$.

La conseqüència és directa: el model no veurà mai cap preu més alt, i als districtes
més cars s'equivocarà sempre. Ho veuràs a la secció 6 i a l'exercici 4. És el tipus
de detall que no surt a cap mètrica i que només es veu mirant les dades.

In [ ]:
dades["ocean_proximity"].value_counts()

La columna de text té cinc valors: `<1H OCEAN` (a menys d'una hora del mar),
`INLAND` (l'interior), `NEAR OCEAN` (a prop del mar), `NEAR BAY` (a la badia de
San Francisco) i `ISLAND`. Aquesta última té **només quatre districtes**: són a
l'illa de Santa Catalina, davant de Los Angeles. Una categoria tan petita dóna
problemes, perquè amb tan pocs exemples qualsevol cosa que en digui un model és
poc de fiar. Hi tornarem a la secció 5.

Ara, on són els districtes cars? Tenim la latitud i la longitud de cadascun.
Dibuixem-les.

In [ ]:
plt.figure(figsize=(8, 7))
mapa = plt.scatter(dades["longitude"], dades["latitude"], c=dades["median_house_value"],
                   s=4, alpha=0.4, cmap="viridis")
plt.colorbar(mapa, label="Preu mitjà ($)")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("On són els districtes cars?")
plt.show()

Dibuixant la longitud contra la latitud apareix **la silueta de Califòrnia**, sense
haver carregat cap mapa: només són dues columnes de números.

I el color ho diu tot: **la costa és cara i l'interior és barat**, amb dues taques
clares, Los Angeles al sud i la badia de San Francisco al centre. Fixa't també en
una altra cosa: **els districtes veïns tenen colors semblants**. Ho necessitarem
dues vegades: per entendre per què la latitud i la longitud funcionen millor juntes
que per separat (exercici 2), i per desconfiar una mica de la partició a l'atzar
(secció 2).

**Extra:** Podem superposar l'scatter plot sobre una imatge del mapa de california i ajustar les axis:

In [ ]:
from pathlib import Path
import tarfile
import urllib.request

plt.figure(figsize=(8, 7))
mapa = plt.scatter(dades["longitude"], dades["latitude"], c=dades["median_house_value"],
                   s=4, alpha=0.4, cmap="viridis")
plt.colorbar(mapa, label="Preu mitjà ($)")
plt.xlabel("Longitud")
plt.ylabel("Latitud")
plt.title("On són els districtes cars?")


# Descarregar la imatge de california i usar-la al fons de l'scatter pot
filename = "california.png"
filepath = Path(f"my_{filename}")
if not filepath.is_file():
    homlp_root = "https://github.com/ageron/handson-mlp/raw/main/"
    url = homlp_root + "images/end_to_end_project/" + filename
    print("Downloading", filename)
    urllib.request.urlretrieve(url, filepath)

california_img = plt.imread(filepath)
axis = -124.55, -113.95, 32.45, 42.05
plt.axis(axis)
plt.imshow(california_img, extent=axis)
###############################

plt.show()

Ens queda una última comprovació: què se n'ha fet dels 207 districtes sense
`total_bedrooms`?

In [ ]:
print(f"Districtes sense total_bedrooms al fitxer sencer: {df['total_bedrooms'].isna().sum()}")
print(f"Districtes sense total_bedrooms a dades:          {dades['total_bedrooms'].isna().sum()}")

**Cap.** Els 207 districtes sense dormitoris han anat a parar tots al test segellat.
Avui ens va bé, perquè `fit()` no accepta valors absents i no haurem de fer res.
Però quan obrim el segell a la UT10 ens hi tornarem a trobar, i per llavors haurem
d'haver après a omplir buits a la **UT3**.

Amb aquestes tres coses al cap (preus tallats, una zona gairebé buida i uns buits
que esperen a la UT10), ja podem posar-nos a treballar.

## 4. Entrenament i validació

Amb el test segellat, tenim un problema: **sobre quines dades comparem els models
d'avui?** No podem fer servir el test, i avaluar sobre l'entrenament no serveix,
com ja saps.

La sortida és tornar a partir el que ens queda. Un tros servirà per entrenar i
l'altre per **validar**: per comparar models i decidir.

Com a entrades farem servir les vuit columnes numèriques. `ocean_proximity` es
queda fora perquè és text, i convertir-la en números és feina de la **UT3**.

In [ ]:
FEATURES = ["median_income", "latitude", "longitude", "housing_median_age",
            "total_rooms", "total_bedrooms", "population", "households"]
OBJECTIU = "median_house_value"

dades = dades.dropna(subset=FEATURES + [OBJECTIU])
print(f"Buits a les dades que ens queden: {dades[FEATURES].isna().sum().sum()}")

train, valid = train_test_split(dades, test_size=0.25, random_state=42)

print(f"Districtes d'entrenament: {len(train)}")
print(f"Districtes de validació:  {len(valid)}")

Ara tenim les dades en tres trossos, i convé tenir clar per a què serveix cadascun:

| Tros | Part del total | Per a què |
|---|---|---|
| **Entrenament** | 60 % | el model hi aprèn amb `fit()` |
| **Validació** | 20 % | nosaltres hi comparem models i decidim |
| **Test segellat** | 20 % | no es toca fins a la UT10 |

El `test_size=0.25` de la segona partició és un quart del que quedava, és a dir,
un 20% del total. Partir en tres trossos una sola vegada és la versió senzilla. A
la **UT7** veurem la validació creuada, que fa aquesta mateixa idea molt més
sòlida.

El `dropna()` treu els districtes on falta alguna mesura, perquè `fit()` no
accepta valors absents. Ja hem vist a la secció 3 que avui no en treu cap, però el
deixem igualment: és un bon hàbit, i no costa res.

## 5. Abans de res, dos models de referència

Ja saps que un número tot sol no vol dir res. Avui en farem servir dos, de
referències:

- **Dir sempre la mitjana.** El `DummyRegressor` de la UT1.
- **Dir el preu mitjà de la seva zona.** És el que diria qualsevol agent
  immobiliari sense saber res d'aprenentatge automàtic: *a la costa és car, a
  l'interior és barat*. Farem servir la columna `ocean_proximity`, que el nostre
  model encara no pot llegir.

Primer preparem una funció que calculi les mètriques del NB 2.1, perquè les
farem servir moltes vegades.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def mesurar(y_real, y_predit):
    return {
        "MAE ($)": mean_absolute_error(y_real, y_predit),
        "RMSE ($)": np.sqrt(mean_squared_error(y_real, y_predit)),
        "R2": r2_score(y_real, y_predit),
    }

Ara calculem el preu mitjà de cada zona **amb les dades d'entrenament**. És
important que surti de l'entrenament i no de la validació: una referència també és
un model, i tampoc pot veure les respostes que després haurà d'endevinar.

In [ ]:
preu_zona = train.groupby("ocean_proximity")[OBJECTIU].mean()
preu_zona.round(-3)

Quatre zones amb preus raonables i una de molt cara, `ISLAND`. És la zona gairebé
buida de la secció 3, i a l'entrenament només n'hi ha dos districtes. Una mitjana
feta amb dos districtes no és gaire de fiar, però la deixem tal com és.

Ara ja podem mesurar les dues referències sobre la validació.

In [ ]:
from sklearn.dummy import DummyRegressor

dummy_mitjana = DummyRegressor(strategy="mean").fit(train[["median_income"]], train[OBJECTIU])

referencies = pd.DataFrame({
    "dir sempre la mitjana": mesurar(valid[OBJECTIU], dummy_mitjana.predict(valid[["median_income"]])),
    "preu mitjà de la zona": mesurar(valid[OBJECTIU], valid["ocean_proximity"].map(preu_zona)),
}).T

referencies.round({"MAE ($)": 0, "RMSE ($)": 0, "R2": 3})

Mira la diferència entre les dues referències. Dir sempre la mitjana s'equivoca
més de **92.000 \$** de mitjana, i el seu R2 és zero, com ha de ser. Dir el preu de
la zona baixa l'error fins als **78.000 \$** i ja té un R2 de 0,23.

Per a nosaltres, **la referència de la zona és el llistó de veritat**. Guanyar el
model que diu la mitjana és fàcil; si el nostre model no guanya el sentit comú,
no val la pena.

---
### Ara tu · Exercici 1: on falla la referència de la zona?

La referència de la zona s'equivoca 78.000 \$ de mitjana, però una mitjana amaga
districtes molt diferents. Mira-ho amb els districtes de validació:

- Calcula l'error de la referència a cada districte: el preu real menys el preu
  de la seva zona, `valid["ocean_proximity"].map(preu_zona)`.
- Quin percentatge de districtes s'equivoca menys de 20.000 \$? I quin, més de
  150.000 \$?
- Mostra els cinc districtes on s'equivoca més, amb el preu real, la zona, els
  ingressos, la latitud i la longitud. Què tenen en comú? Et sona d'alguna cosa
  de la secció 3?

Pista: guarda l'error en valor absolut (`.abs()`) en una columna nova d'una còpia
de `valid`, i després fes servir `nlargest(5, "nom_de_la_columna")`.

In [ ]:
# Escriu aquí el teu codi

## 6. Una sola variable: els ingressos

Comencem com al NB 2.1, amb una recta. L'entrada serà `median_income`, els
ingressos mitjans del districte. És la candidata més evident: als barris on es
guanya més, les cases solen ser més cares.

In [ ]:
from sklearn.linear_model import LinearRegression

model_1 = LinearRegression().fit(train[["median_income"]], train[OBJECTIU])

print(f"Pendent:             {model_1.coef_[0]:,.0f} $ per cada 10.000 $ d'ingressos")
print(f"Ordenada a l'origen: {model_1.intercept_:,.0f} $")
print()
resultat = mesurar(valid[OBJECTIU], model_1.predict(valid[["median_income"]]))
print("Sobre la validació:")
print(f"  MAE:  {resultat['MAE ($)']:,.0f} $")
print(f"  RMSE: {resultat['RMSE ($)']:,.0f} $")
print(f"  R2:   {resultat['R2']:.3f}")

Llegim el model en veu alta, com vam fer amb els pingüins:

> **El preu mitjà del districte serà de 44.000 \$, més 42.000 \$ per cada 10.000 \$
> d'ingressos mitjans.**

(Python escriu els milers amb comes, a l'anglesa: `42,010` vol dir 42.010.)

I el resultat? Un MAE de **63.000 \$**, contra els 78.000 \$ de la referència de la
zona. **Una sola columna ja guanya el sentit comú, i per molt.** L'R2 passa de
0,23 a 0,48: els ingressos del veïnat expliquen gairebé la meitat de les
diferències de preu entre districtes.

In [ ]:
ingressos = pd.DataFrame({"median_income": np.linspace(0, 15, 50)})

plt.figure(figsize=(7, 5))
plt.scatter(valid["median_income"], valid[OBJECTIU], alpha=0.2, s=10, label="districtes de validació")
plt.plot(ingressos["median_income"], model_1.predict(ingressos), color="tab:red", linewidth=2, label="recta apresa")
plt.xlabel("Ingressos mitjans (desenes de milers de $)")
plt.ylabel("Preu mitjà de l'habitatge ($)")
plt.legend()
plt.show()

El núvol puja en diagonal i la recta el segueix prou bé. Però mira-la pel costat
dret: **continua pujant per sobre dels 500.001 \$**, on no hi ha cap districte
perquè les dades estan tallades. Per als districtes més rics, el model prediu
preus que les dades no poden tenir mai. Ho tornaràs a trobar a l'exercici 4.

Fixa't també en com n'és de gruixut el núvol. Amb els mateixos ingressos hi ha
districtes de 100.000 \$ i de 400.000 \$. Aquesta és la feina que queda per a la
resta de variables.

## 7. Afegir variables d'una en una

Ara la pregunta del dia. Afegirem variables en un ordre que té sentit intuïtiu:
primer els ingressos, després on és el districte, i al final com són els
habitatges i quanta gent hi viu. A cada pas entrenarem un model nou i n'apuntarem
els errors.

In [ ]:
resultats = []

for n in range(1, len(FEATURES) + 1):
    variables = FEATURES[:n]
    model = LinearRegression().fit(train[variables], train[OBJECTIU])

    resultats.append({
        "variables": n,
        "afegida": variables[-1],
        "MAE entrenament": mean_absolute_error(train[OBJECTIU], model.predict(train[variables])),
        "MAE validació": mean_absolute_error(valid[OBJECTIU], model.predict(valid[variables])),
        "R2 validació": r2_score(valid[OBJECTIU], model.predict(valid[variables])),
    })

resultats = pd.DataFrame(resultats)
resultats.round({"MAE entrenament": 0, "MAE validació": 0, "R2 validació": 3})

Llegeix la columna del MAE de validació de dalt a baix. Hi ha tres coses a veure:

**La primera variable fa la major part de la feina.** Passar de no tenir cap dada
(92.000 \$ amb la mitjana) a tenir els ingressos (63.000 \$) és un salt enorme. Les
set variables que vénen després, totes juntes, baixen l'error uns 12.000 \$ més.

**Una variable pot semblar inútil i no ser-ho.** Mira la latitud: tota sola,
afegida als ingressos, gairebé no mou res, menys de 1.000 \$. En canvi la
longitud, just després, baixa l'error més de 7.000 \$. No és que la longitud sigui
millor que la latitud. És que **juntes diuen on és el districte**, i cadascuna per
separat no ho diu. Ho comprovaràs a l'exercici 2.

**Algunes variables no aporten res.** La darrera, `households`, deixa l'error
pràcticament igual. Pensa-hi: un districte amb el doble de llars té, més o menys,
el doble de dormitoris, i el model ja tenia `total_bedrooms`. **Una variable que
repeteix el que ja saps no afegeix informació.**

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(resultats["variables"], resultats["MAE validació"], marker="o", label="model lineal")
plt.axhline(referencies.loc["preu mitjà de la zona", "MAE ($)"], color="gray", linestyle="--", label="preu mitjà de la zona")
plt.xticks(resultats["variables"], resultats["afegida"], rotation=45)
plt.xlabel("Última variable afegida")
plt.ylabel("MAE de validació ($)")
plt.legend()
plt.show()

A la gràfica es veu tot junt: el primer punt ja és molt per sota de la referència
de la zona, el salt de la longitud és el més gran de la resta, i l'últim tram és
pla.

Fixa't també en una cosa de la taula: **els errors d'entrenament i de validació
són molt semblants**. El model s'equivoca pràcticament igual amb districtes que ha
vist i amb districtes que no. És el senyal que no està *memoritzant* res, i és el
que esperem d'una regressió lineal. Al NB 2.3 veurem un model que sí que
memoritza, i com es nota en aquestes dues columnes.

Afegir variables, a més, **té costos** encara que el número millori:

- Per fer una predicció, necessites **totes** les variables de cada districte. Si
  en un districte falta el nombre de dormitoris (i ja has vist que passa), el
  model de vuit variables no pot predir.
- Un model amb vuit coeficients és molt més difícil d'explicar que un amb un.
- Com més variables, més possibilitats que el model s'ajusti a casualitats de les
  dades d'entrenament. Ho veurem de prop al NB 2.3.

---
### Ara tu · Exercici 2: la latitud i la longitud, per separat

Entrena tres models, cadascun amb una sola entrada o dues: només `latitude`,
només `longitude`, i totes dues juntes. Quin MAE i quin R2 de validació té
cadascun? Compara'ls amb la referència de dir sempre la mitjana.

Després torna a dibuixar el mapa de la secció 3: longitud a l'eix horitzontal,
latitud al vertical, i el color segons el preu (`c=train[OBJECTIU]`). Per què creus
que cadascuna de les dues coordenades, tota sola, prediu tan malament, i en canvi
juntes funcionen força millor?

Pista: fixa't en la direcció que fa la costa de Califòrnia. Moure's cap al nord,
t'acosta o t'allunya del mar?

In [ ]:
# Escriu aquí el teu codi

## 8. Llegir els coeficients quan n'hi ha molts

In [ ]:
model_tot = LinearRegression().fit(train[FEATURES], train[OBJECTIU])

coeficients = pd.Series(model_tot.coef_, index=FEATURES).round(2)
print(f"Ordenada a l'origen: {model_tot.intercept_:,.0f} $")
coeficients

Cada coeficient es llegeix gairebé com al NB 2.1, però amb una coletilla
important: **si la resta de variables no canvien**. Per exemple, el coeficient de
`latitude`, uns −42.700 \$, diu que un districte un grau més al nord (uns 111 km)
val 42.700 \$ menys *que un altre amb els mateixos ingressos, la mateixa
longitud, la mateixa antiguitat i la mateixa mida*.

Mira la latitud i la longitud juntes: tenen un coeficient negatiu gairebé idèntic.
Traduït: el preu baixa quan et mous cap al nord i quan et mous cap a l'est. Com
que la costa de Califòrnia fa una diagonal, **anar cap al nord-est és allunyar-se
del mar**. El model ha descobert la costa sense que ningú li digués on és.

I ara el parany. Seria temptador pensar que la variable amb el coeficient més gran
és la més important. **No es pot fer aquesta lectura**, per dos motius:

- **Les variables tenen unitats diferents.** Un grau de latitud, una desena de
  milers de dòlars d'ingressos, un any d'antiguitat i un dormitori no són
  comparables. Si els ingressos vinguessin en dòlars en lloc de desenes de milers,
  el seu coeficient seria deu mil vegades més petit sense que el model canviés
  gens.
- **Les variables s'assemblen entre elles.** `total_rooms`, `total_bedrooms`,
  `population` i `households` mesuren, totes quatre, **la mida del districte**, i
  pugen i baixen juntes. Quan diverses variables diuen gairebé el mateix, el model
  es reparteix el pes entre elles d'una manera que pot semblar estranya. Mira-ho:
  segons el model, cada habitació de més fa *baixar* el preu, i cada dormitori de
  més el fa *pujar* més de cent dòlars. No té cap sentit que tenir més habitacions
  abarateixi un districte. És el model compensant una variable amb les altres.

Ah, i l'ordenada a l'origen, uns **−3,6 milions** de dòlars? No té cap sentit
físic: és el preu que diria el model per a un districte a latitud 0 i longitud 0,
al mig de l'oceà davant de l'Àfrica, sense ingressos ni cases. Cap districte real
s'hi acosta, i per això el número pot ser tan absurd.

Com posar totes les variables a la mateixa escala és contingut de la **UT3**, i
com mesurar de veritat quina variable és més important, de la **UT5**.

---
### Ara tu · Exercici 3: quatre mides o dues?

Entrena el model de totes les variables però **sense `total_bedrooms` ni
`households`**, i mira'n els coeficients i el MAE de validació. Què li ha passat
al coeficient de `total_rooms`? Ara té més sentit? I el MAE, ha canviat gaire?
Què et diu això sobre la necessitat de tenir les quatre mides al model?

Pista: `[f for f in FEATURES if f not in ["total_bedrooms", "households"]]` et
dóna la llista de variables sense les que vols treure.

In [ ]:
# Escriu aquí el teu codi

## 9. Per què la latitud sola no ajuda?

Hem vist que la latitud, tota sola, gairebé no millora el model. Mirem-ho
dibuixat. A més de la millor recta, hi afegim una línia negra amb el preu mitjà
dels districtes de cada franja de mig grau de latitud, perquè amb vint mil punts
el núvol no deixa veure la forma.

In [ ]:
recta_lat = LinearRegression().fit(train[["latitude"]], train[OBJECTIU])
latituds = pd.DataFrame({"latitude": np.linspace(32.5, 42, 100)})

# (latitud * 2).round() / 2 arrodoneix cada latitud al mig grau més proper
mitjana_franja = train.groupby((train["latitude"] * 2).round() / 2)[OBJECTIU].mean()

plt.figure(figsize=(8, 4))
plt.scatter(train["latitude"], train[OBJECTIU], alpha=0.05, s=8, label="districtes d'entrenament")
plt.plot(mitjana_franja.index, mitjana_franja.values, color="black", marker="o", label="preu mitjà de cada franja")
plt.plot(latituds["latitude"], recta_lat.predict(latituds), color="tab:red", linewidth=2, label="la millor recta")
plt.xlabel("Latitud (graus nord)")
plt.ylabel("Preu mitjà de l'habitatge ($)")
plt.legend()
plt.show()

La línia negra té una forma molt clara: **dues muntanyes**. La primera, cap als
33-34 graus, és la zona de Los Angeles i San Diego. La segona, cap als 37-38, és
la badia de San Francisco. Entremig hi ha la vall central, i al nord la
Califòrnia rural, molt més barates. **Però això no és una recta.** La millor recta
que es pot traçar per aquest núvol baixa una mica de sud a nord i no s'assembla
gens a la forma de les dades: a les dues ciutats es queda curta, i a la vall
central i al nord es passa.

Això no vol dir que la latitud no tingui informació. Vol dir que **el nostre model
només sap dibuixar rectes**. Hi ha dues sortides: donar-li al model una manera de
corbar-se, o donar-li la informació ja preparada perquè una recta la pugui fer
servir (per exemple, la distància a la ciutat gran més propera). La primera és el
tema del NB 2.3; la segona, de la UT3.

Una última observació, que al NB 2.3 tindrà importància: fixa't en com n'és de
gruixut el núvol gris al voltant de la línia negra. A la mateixa latitud hi ha
districtes de 50.000 \$ i de 500.000 \$. La forma hi és, però hi ha **molt de
soroll** al seu voltant.

## 10. Exercicis finals

Els tres exercicis *Ara tu* els has fet pel camí, cadascun lligat a una secció.
Aquests tres últims et demanen fer servir el notebook sencer.

**4. On s'equivoca més?** A l'exercici 1 has buscat els pitjors districtes de la
referència de la zona. Ara mira-ho amb el model de totes les variables
(`model_tot`): calcula el MAE de validació **per a cada zona** d'`ocean_proximity`,
i després separa els districtes tallats a 500.001 \$ de la resta. On s'equivoca
més? Hi ha cap predicció negativa? Tens alguna hipòtesi de per què?
Pista: afegeix una columna d'error absolut a una còpia de `valid` i fes servir
`groupby("ocean_proximity")`.

**5. El parany del percentatge.** Calcula la desviació percentual (NB 2.1) del
`model_1` sobre la validació. Després calcula-la per separat per als districtes
de menys de 100.000 \$ i per als de més de 300.000 \$, i fes el mateix amb el MAE.
Quin dels dos grups té més error en dòlars? I en percentatge? Què et diu això
sobre quina mètrica triar quan els valors a predir són de mides molt diferents?

**6. Una altra validació.** Repeteix la partició d'entrenament i validació de la
secció 4 canviant el `random_state` per 0, 1 i 7, i calcula cada vegada el MAE de
validació del model de totes les variables. Surt sempre igual? Què vol dir això
sobre la confiança que podem tenir en una sola xifra de validació?
**Compte**: canvia només el de la segona partició. El de la primera és el pany
del test segellat i no es toca mai.

## 11. Per al debat de classe

El model de vuit variables s'equivoca uns 51.000 \$ de mitjana, i el d'una sola
variable, uns 63.000 \$. El preu mitjà d'un districte és d'uns 180.000 \$.

- Si fossis una immobiliària, faries servir aquest model per posar preu a una
  casa? I per decidir en quins barris obrir una oficina? Per què la resposta pot
  ser diferent?
- 51.000 \$ d'error, és molt o poc? Per a qui?
- El model no fa servir `ocean_proximity`, ni sap res de les habitacions *per
  llar* (només en té els totals del districte), ni pot predir per sobre de
  500.001 \$.
  Què creus que milloraria més el model: afegir més variables, preparar millor les
  que ja té, o arreglar les dades?